# Receptor refinement geometry — before vs after the signed graph

Our thesis is that the signed bipartite GNN **improves the receptor representation from
binding data**. This notebook makes that visible. It takes the M2OR cold-molecule
(*inductive*) signed-graph run — five trained `_SignedSage` models, one per seed
(folds 42–46) — and shows, as 2D PCA scatters (matplotlib), the receptor cloud:

1. the **initial** cloud — raw mean-pooled ESM-1b, before any message passing;
2. **five refined** clouds — the graph-enriched receptor vectors `z[PROT]`, one per model.

Each point is a receptor; **unknown receptors — unannotated / non-human sequences and
point-mutants — are dropped** (from the plots and every count), and the survivors (human
wild-type ORs) are coloured by **OR family** (~13 colours). If the graph reorganises
receptors by their ligand profile, the refined clouds should sharpen family structure
relative to raw ESM. The analysis cells quantify that (silhouette / kNN purity), check
whether the five models agree on the geometry (CKA), rank which receptors move most toward
their family, and finally test whether the refined geometry aligns with **functional
(binding-profile) proximity** better than raw ESM.


In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)   # orbind resolves its data/ paths relative to CWD, so run from the repo root

import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from orbind.regimes import full_full_pairs, load_split
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict,
    _train_one, MOL, PROT,
)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# --- which run's models to visualise -------------------------------------
# M2OR cold-molecule (inductive) signed-graph run: 5 seeds (folds 42-46), each a
# trained _SignedSage. If the run saved checkpoints under
# RUN_DIR/checkpoints/repeat_{seed}/gnn_{CLS_NAME}_model0.pt they are LOADED; else the
# notebook REPRODUCES each model from the exact same config/protocol (the graph is tiny,
# ~seconds/seed), so it runs anywhere the embeddings live.
RUN_DIR  = ROOT / "results/ensemble_logs/m2or-inductive-chemberta-fixed/gnn_signed_q99_greedy_clsprotmol"
CLS_NAME = "cls"                      # source name -> checkpoint file gnn_{CLS_NAME}_model0.pt
REGIME   = "inductive_molecule_v5"
SEEDS    = [42, 43, 44, 45, 46]

# graph config = the run's `cls=gnn_signed` source, verbatim (q99 / greedy / emit=prot)
PROT_NPZ = DATA / "embeddings/proteins/esm1b_650m_mean.npz"
MOL_NPZ  = DATA / "embeddings/molecules/gin_supervised_contextpred_all_m2or.npz"
GRAPH_KW = dict(q=0.99, criterion="greedy_pair_cover", edge_threshold=0.0,
                k_mode="coverage_quantile", emit="prot", n_models=1, hidden=256)

N_FAMILY_COLORS = 13   # keep the top (N-1) OR families by count + one "other" bucket

## 1 · Load / reproduce the five models and pull the embeddings

For each seed we rebuild the exact inductive universe and message-passing graph, load the
trained encoder (or retrain it), then read off two things: the **initial** node features
(raw ESM, shared across seeds) and the **refined** receptor vectors `z[PROT]` (one set per
seed). Receptors are keyed by name so everything aligns across seeds.


In [ ]:
pairs = full_full_pairs()
ext = GnnSignedExtractor(name=CLS_NAME, protein_path=str(PROT_NPZ),
                         molecule_path=str(MOL_NPZ), **GRAPH_KW)

initial, refined = None, {}
for seed in SEEDS:
    tr, va, te = load_split(REGIME, seed)
    # keep only rows this graph covers (protein + molecule embedding present) --
    # the run does this via `--on-missing drop` before building the universe.
    tr, va, te = (i[ext.covered(pairs, i)] for i in (tr, va, te))
    all_idx = np.concatenate([tr, va, te])
    mol_to_i, prot_to_i, x_mol, x_prot = _build_universe(pairs, all_idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(pairs, tr, mol_to_i, prot_to_i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pos_eidx, neg_eidx = _edge_index_dict(pos, neg)

    tdf = pairs.iloc[tr]
    mi = tdf["inchikey"].map(mol_to_i).to_numpy()
    pi = tdf["receptor"].map(prot_to_i).to_numpy()
    y  = tdf["label"].to_numpy(dtype=np.float32)

    ckpt = RUN_DIR / "checkpoints" / f"repeat_{seed}" / f"gnn_{CLS_NAME}_model0.pt"
    z_mol, z_prot, model = _train_one(
        ext._build_model, x_mol, x_prot, pos_eidx, neg_eidx, mi, pi, y,
        ext._hp(seed), DEVICE, checkpoint_path=(ckpt if ckpt.exists() else None))

    refined[seed] = {r: z_prot[i] for r, i in prot_to_i.items()}
    if initial is None:
        initial = {r: x_prot[i].numpy() for r, i in prot_to_i.items()}
    print(f"seed {seed}: {'loaded' if ckpt.exists() else 'reproduced':10s} "
          f"| {len(prot_to_i):4d} receptors | {len(pos[0])}+/{len(neg[0])}- MP edges "
          f"| refined dim {z_prot.shape[1]}")

REC = sorted(set(initial).intersection(*[set(refined[s]) for s in SEEDS]))
init_mat     = np.stack([initial[r] for r in REC])
refined_mats = {s: np.stack([refined[s][r] for r in REC]) for s in SEEDS}
print(f"\naligned on {len(REC)} receptors | init {init_mat.shape} | refined {refined_mats[SEEDS[0]].shape}")

## 2 · Drop unknown receptors, colour the rest by OR family

The receptor names are raw sequences; `bw_ref_used_curated.csv` annotates the **human,
wild-type** ORs with their gene → OR family (same recipe as
`protein_embeddings/pca19_family_3d.ipynb`). Every receptor **without** a real OR family —
unannotated / non-human sequences, and the point-mutants (a mutant is a second sequence
sharing one UniProt id) — is **dropped from the plots and all downstream counts**. The
survivors are coloured by family: the top `N_FAMILY_COLORS-1` by size, the rare real
families folded into `other` (~13 colours total).


In [ ]:
bw = pd.read_csv(DATA / "processed/bw_ref_used_curated.csv")
seq2ref = bw.set_index("receptor")["ref_entry"].to_dict()

def or_family(rec):
    slug = seq2ref.get(rec, "unk").replace("_human", "")
    m = re.match(r"^(or?\d+)", slug)
    return m.group(1).upper() if m else None   # None -> not an annotated OR (unknown/mutant/non-human)

# --- drop unknown receptors from EVERYTHING (viz + counts) -------------------
fam_full = {r: or_family(r) for r in REC}
kept = [r for r in REC if fam_full[r] is not None]

n_mut = None                     # of the dropped, how many are true point-mutants?
try:
    master = pd.read_csv(DATA / "processed/proteins/receptor_master.csv")
    inuni  = master[master["sequence"].isin(REC)]
    dup    = inuni.groupby("uniprot_id")["sequence"].transform("nunique") > 1
    n_mut  = len(set(inuni.loc[dup & (~inuni["in_curated"]), "sequence"]) & set(REC))
except Exception as e:
    print("mutant count skipped:", e)

print(f"receptors: {len(REC)} -> kept {len(kept)} annotated human OR (dropped {len(REC)-len(kept)} "
      f"unknown" + (f", incl. {n_mut} point-mutants" if n_mut is not None else "") + ")")

REC          = kept                                     # <-- everything below uses the filtered set
init_mat     = np.stack([initial[r] for r in REC])
refined_mats = {s: np.stack([refined[s][r] for r in REC]) for s in SEEDS}

fam_all = {r: or_family(r) for r in REC}
counts  = pd.Series(list(fam_all.values())).value_counts()
top     = list(counts.index[:N_FAMILY_COLORS - 1])
fam     = np.array([fam_all[r] if fam_all[r] in top else "other" for r in REC])

fam_levels = [f for f in top if f in set(fam)] + (["other"] if "other" in fam else [])
cmap       = plt.get_cmap("tab20", len(fam_levels))
color_map  = {f: cmap(i) for i, f in enumerate(fam_levels)}
if "other" in color_map:
    color_map["other"] = (0.75, 0.75, 0.75, 1.0)
print(f"{len(fam_levels)} colour groups (top families + rare 'other'):", fam_levels)
counts.head(N_FAMILY_COLORS)

## 3 · Initial cloud — raw ESM-1b

The starting point every model shares: mean-pooled ESM-1b, PCA-projected to 2D.


In [ ]:
from sklearn.decomposition import PCA

def to2d(M):
    return PCA(n_components=2, random_state=0).fit_transform(np.asarray(M, dtype=np.float64))

def scatter2d(ax, coords, title):
    for f in fam_levels:
        m = fam == f
        if not m.any():
            continue
        ax.scatter(coords[m, 0], coords[m, 1], s=14, alpha=0.8,
                   color=color_map[f], label=f, edgecolors="none")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.tick_params(labelsize=7)

fig, ax = plt.subplots(figsize=(7.5, 6))
scatter2d(ax, to2d(init_mat), f"Initial receptors — raw ESM-1b ({init_mat.shape[1]}d) -> PCA-2")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7,
          frameon=False, title="OR family", title_fontsize=8)
fig.tight_layout()
plt.show()

## 4 · Refined clouds — one per model (5 seeds)

The graph-enriched receptor vectors, five independently-seeded encoders. Compare their
family organisation to the raw cloud above and to each other (the sixth panel is left
blank).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, s in zip(axes.ravel(), SEEDS):
    scatter2d(ax, to2d(refined_mats[s]),
              f"Refined — seed {s} ({refined_mats[s].shape[1]}d) -> PCA-2")
axes.ravel()[-1].axis("off")
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center", bbox_to_anchor=(0.83, 0.25),
           fontsize=8, frameon=False, title="OR family", ncol=2)
fig.suptitle("Graph-refined receptor clouds (PCA-2), one per seed", y=1.0, fontsize=12)
fig.tight_layout()
plt.show()

## 5 · Does refinement sharpen family structure?

Two label-aware geometry scores on the **full-dimensional** vectors (not the 2D
projection), over real families only (the `other` bucket dropped): the **silhouette** of
the family clustering, and **kNN purity** (share of each receptor's 5 nearest neighbours in
the same family). Higher = families sit together more cleanly. We compare raw ESM against
each refined model.


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

keep = fam != "other"

def knn_purity(X, labels, k=5):
    k = min(k, len(X) - 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    idx = nn.kneighbors(return_distance=False)[:, 1:]
    return float(np.mean([np.mean(labels[nbrs] == labels[i]) for i, nbrs in enumerate(idx)]))

def geo_scores(X):
    Xk, lk = X[keep], fam[keep]
    return silhouette_score(Xk, lk), knn_purity(Xk, lk)

rows = [("initial (raw ESM)", *geo_scores(init_mat))]
rows += [(f"refined seed {s}", *geo_scores(refined_mats[s])) for s in SEEDS]
geo = pd.DataFrame(rows, columns=["embedding", "silhouette", "knn_purity_k5"])

ref = geo.iloc[1:]
print(f"raw ESM: silhouette {geo.iloc[0].silhouette:+.3f}, kNN purity {geo.iloc[0].knn_purity_k5:.3f}")
print(f"refined: silhouette {ref.silhouette.mean():+.3f} +/- {ref.silhouette.std():.3f}, "
      f"kNN purity {ref.knn_purity_k5.mean():.3f} +/- {ref.knn_purity_k5.std():.3f} (mean over 5 seeds)")
display(geo.style.format({"silhouette": "{:+.3f}", "knn_purity_k5": "{:.3f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(geo)), geo.knn_purity_k5, color=["#888888"] + ["#4c78a8"] * len(SEEDS))
ax.axhline(geo.iloc[0].knn_purity_k5, ls=":", c="#888888", label="raw ESM")
ax.set_xticks(range(len(geo)))
ax.set_xticklabels(geo.embedding, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("kNN purity (k=5)")
ax.set_title("kNN family purity: raw ESM vs 5 refined models")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 6 · Do the five models agree on the geometry?

Linear **CKA** between every pair of embedding spaces (representation similarity that is
invariant to rotation/scaling and works across dimensionalities). If the five seeds
discover the *same* refined geometry — not seed noise — the refined-vs-refined block should
be high; the init-vs-refined column shows how far each model moved from raw ESM.


In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(0)
    Y = Y - Y.mean(0)
    return float((np.linalg.norm(Y.T @ X) ** 2) /
                 (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)))

mats = {"init": init_mat, **{f"s{s}": refined_mats[s] for s in SEEDS}}
keys = list(mats)
C = np.array([[linear_cka(mats[a], mats[b]) for b in keys] for a in keys])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(C, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys)
ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys)
for i in range(len(keys)):
    for j in range(len(keys)):
        ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center",
                color="w" if C[i, j] < 0.6 else "k", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Linear CKA — raw init vs 5 refined models")
fig.tight_layout()
plt.show()

rr = C[1:, 1:][np.triu_indices(len(SEEDS), 1)]
print(f"refined<->refined CKA: {rr.mean():.3f} +/- {rr.std():.3f}  |  "
      f"init<->refined CKA: {C[0, 1:].mean():.3f} +/- {C[0, 1:].std():.3f}")

## 7 · Which receptors move most toward their family?

Per receptor, the cosine similarity to its own family centroid, raw vs refined (averaged
over the five models). A positive `delta` means the graph pulled that receptor closer to
its family — the receptors where ligand-profile refinement adds the most that raw sequence
identity missed.


In [ ]:
def centroid_cos(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    sim = np.full(len(REC), np.nan)
    for f in fam_levels:
        if f == "other":
            continue
        m = fam == f
        if m.sum() < 2:
            continue
        c = Xn[m].mean(0)
        c = c / (np.linalg.norm(c) + 1e-9)
        sim[m] = Xn[m] @ c
    return sim

base     = centroid_cos(init_mat)
ref_mean = np.nanmean(np.stack([centroid_cos(refined_mats[s]) for s in SEEDS]), axis=0)
mv = (pd.DataFrame({"receptor": REC, "family": fam, "cos_init": base,
                    "cos_refined": ref_mean, "delta": ref_mean - base})
      .query("family != 'other'").dropna().sort_values("delta", ascending=False))

print(f"mean cosine-to-family-centroid: raw {np.nanmean(base):.3f} -> refined "
      f"{np.nanmean(ref_mean):.3f}  (mean delta {mv['delta'].mean():+.3f})")
display(pd.concat([mv.head(10), mv.tail(5)])
        .style.format({"cos_init": "{:.3f}", "cos_refined": "{:.3f}", "delta": "{:+.3f}"})
        .hide(axis="index"))

## 8 · Embedding geometry vs functional (binding-profile) proximity

Our long-standing *functional* similarity of two receptors: compare their **binding
profiles over the whole molecule set**. Each receptor becomes a signed vector over every
M2OR molecule (+1 bound / -1 not bound / 0 untested); functional similarity is the cosine
of those vectors, which counts only co-tested molecules. We then ask whether the graph
pulls the embedding geometry **toward** this functional structure: for each receptor, the
overlap between its top-10 functional neighbours and its top-10 neighbours in raw ESM vs
each refined space, plus a Spearman correlation of the full similarity matrices (over
receptor pairs sharing >=5 molecules).

*Caveat:* the refined embedding is trained on train binding edges, so a rise here is the
mechanism working as intended (the graph encodes binding), not an independent oracle — but
raw ESM sees none of it, so the gap measures how much functional signal refinement injects.


In [ ]:
# signed binding profile over EVERY molecule, for the kept receptors
sub   = pairs[pairs["receptor"].isin(set(REC))][["receptor", "inchikey", "label"]]
mols  = sorted(sub["inchikey"].unique())
r_idx = {r: i for i, r in enumerate(REC)}
m_idx = {m: j for j, m in enumerate(mols)}
g = sub.groupby(["receptor", "inchikey"], as_index=False)["label"].mean()
B = np.zeros((len(REC), len(mols)), np.float32)
B[g["receptor"].map(r_idx).to_numpy(), g["inchikey"].map(m_idx).to_numpy()] = \
    np.where(g["label"].to_numpy() >= 0.5, 1.0, -1.0)

tested = B != 0
Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
F  = Bn @ Bn.T                                                   # functional similarity (cosine)
cotest = tested.astype(np.float32) @ tested.astype(np.float32).T  # shared-molecule counts
print(f"binding profile: {B.shape[0]} receptors x {B.shape[1]} molecules | "
      f"median molecules tested/receptor: {int(np.median(tested.sum(1)))}")

In [ ]:
from scipy.stats import spearmanr

def emb_sim(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T

def topk_idx(S, k):
    S = S.copy(); np.fill_diagonal(S, -np.inf)
    return np.argpartition(-S, k, axis=1)[:, :k]

def overlap_at_k(Sfun, Semb, k=10):
    A, Bk = topk_idx(Sfun, k), topk_idx(Semb, k)
    return float(np.mean([len(set(A[i]) & set(Bk[i])) / k for i in range(len(A))]))

iu     = np.triu_indices(len(REC), 1)
mask   = cotest[iu] >= 5                       # only pairs with enough shared molecules
f_flat = F[iu][mask]

def align(X, k=10):
    S = emb_sim(X)
    return overlap_at_k(F, S, k), spearmanr(f_flat, S[iu][mask]).correlation

rows = [("raw ESM", *align(init_mat))] + [(f"refined s{s}", *align(refined_mats[s])) for s in SEEDS]
fa = pd.DataFrame(rows, columns=["embedding", "overlap@10_functional", "spearman_functional"])
r = fa.iloc[1:]
print(f"raw ESM : overlap@10 {fa.iloc[0,1]:.3f} | Spearman {fa.iloc[0,2]:+.3f}")
print(f"refined : overlap@10 {r.iloc[:,1].mean():.3f} +/- {r.iloc[:,1].std():.3f} | "
      f"Spearman {r.iloc[:,2].mean():+.3f} +/- {r.iloc[:,2].std():.3f} (mean over 5 seeds)")
display(fa.style.format({"overlap@10_functional": "{:.3f}", "spearman_functional": "{:+.3f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(fa)), fa["overlap@10_functional"], color=["#888888"] + ["#e4794c"] * len(SEEDS))
ax.axhline(fa.iloc[0, 1], ls=":", c="#888888", label="raw ESM")
ax.set_xticks(range(len(fa))); ax.set_xticklabels(fa.embedding, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("overlap@10 with functional neighbours")
ax.set_title("Do embeddings agree with binding-profile (functional) neighbours?")
ax.legend(frameon=False); fig.tight_layout(); plt.show()

**Read the three analyses together.** If silhouette/kNN purity rise from raw to refined,
the graph is genuinely tightening family structure; the CKA block says whether the five
seeds agree that this is a real, reproducible geometry rather than per-seed noise; and the
mover table names the receptors that benefit most — a shortlist of where binding-profile
information corrects sequence-only identity; and section 8 checks the geometry directly
against functional (binding-profile) proximity, where the refined spaces should sit closer
to the functional neighbour structure than raw ESM. This is the receptor-side improvement
the paper argues the field has under-explored, made visible.
